# Inverse Design: Grating Coupler on 220nm SOI

**Pipeline:** theta -> density -> Layer -> structure -> mode solve -> optimize -> surgery -> DRC -> GDS


## Step 1: Environment Setup


In [ ]:
# Install dependencies
!pip install -q hyperwave-community matplotlib

# Imports
import numpy as np
import hyperwave_community as hwc

## Step 2: Device Configuration


In [ ]:
# --- Material refractive indices ---
n_si = 3.48          # Silicon at 1550nm
n_sio2 = 1.44        # SiO2 cladding
n_air = 1.0          # Air / top cladding

# --- Wavelength and grid ---
wavelength = 1.55    # um (1550nm)
fdtd_grid = 0.035    # um (35nm FDTD grid spacing)
theta_grid = fdtd_grid / 2  # 17.5nm design grid (2x oversampling)

# --- Device stack ---
wg_height = 0.220    # um (full silicon thickness)
etch_depth = 0.110   # um (partial etch for grating)
slab_height = wg_height - etch_depth  # 110nm remaining slab
box_thickness = 2.0  # um (buried oxide)
clad_thickness = 2.0 # um (top cladding)

# --- Waveguide ---
wg_width = 0.500     # um (single-mode strip waveguide)

# --- Fiber parameters ---
fiber_angle = 14.5   # degrees from surface normal
beam_waist = 5.2     # um (SMF-28 mode field diameter / 2)

# --- Design region ---
design_length = 35.0 # um (grating coupler length)
design_width = 35.0  # um (grating coupler width)

# --- Feature size constraints (integer pixel units) ---
# density_radius is set per design layer when calling hwc.density()
# R=6 pixels -> min feature = (2*6+1) * 17.5nm = 228nm
drc_disk_radius = 3          # disk(r=3), DRC check = (2*3+1)*17.5nm = 122nm

print(f"Device: {int(wg_height*1000)}nm SOI, {int(etch_depth*1000)}nm etch, {int(wavelength*1000)}nm")
print(f"Grid: {int(fdtd_grid*1000)}nm FDTD, {theta_grid*1000:.1f}nm theta")
print(f"Density filter: R=6 -> min feature = {(2*6+1)*theta_grid*1000:.0f}nm (set per-layer)")
print(f"DRC disk:       r={drc_disk_radius} -> min check = {(2*drc_disk_radius+1)*theta_grid*1000:.0f}nm")

## Step 3a: Grid Dimensions


In [ ]:
# --- Grid dimensions ---
nx = int(design_length / theta_grid)   # x pixels (propagation direction)
ny = int(design_width / theta_grid)    # y pixels (transverse direction)

print(f"Grid: {nx} x {ny} pixels ({design_length} x {design_width} um)")


## Step 3b: Initialize Theta (Design Variables)

Theta is the raw optimization variable. Each pixel is a value in [0, 1] that the optimizer will tune.


In [ ]:
import jax.numpy as jnp

# Slab layer: start fully solid (silicon)
theta_slab = jnp.ones((nx, ny), dtype=jnp.float32)

# Etch layer: start at 0.5 (grayscale, optimizer will push toward 0 or 1)
theta_etch = jnp.full((nx, ny), 0.5, dtype=jnp.float32)

print(f"theta_slab shape: {theta_slab.shape}, range: [{float(theta_slab.min())}, {float(theta_slab.max())}]")
print(f"theta_etch shape: {theta_etch.shape}, range: [{float(theta_etch.min())}, {float(theta_etch.max())}]")


## Step 3c: Apply Density Filter

The density filter enforces minimum feature size. `radius=6` at 17.5nm grid gives ~228nm min feature.


In [ ]:
density_slab = hwc.density(theta_slab, radius=6)
density_etch = hwc.density(theta_etch, radius=6)

print(f"density_slab shape: {density_slab.shape}")
print(f"density_etch shape: {density_etch.shape}")


## Step 3d: Build Layer Objects

Each layer maps density to permittivity. Design layers interpolate between two materials; non-design layers are fixed.


In [ ]:
h_slab = int(round(slab_height / fdtd_grid))
h_etch = int(round(etch_depth / fdtd_grid))

# Design layers (density controls material distribution)
slab_layer = hwc.Layer(density_slab, permittivity_values=(n_sio2**2, n_si**2), layer_thickness=h_slab)
etch_layer = hwc.Layer(density_etch, permittivity_values=(1.0, n_si**2), layer_thickness=h_etch)

# Non-design layers (fixed permittivity)
substrate_layer = hwc.Layer(jnp.zeros((nx, ny)), permittivity_values=n_si**2, layer_thickness=int(0.5/fdtd_grid))
box_layer = hwc.Layer(jnp.zeros((nx, ny)), permittivity_values=n_sio2**2, layer_thickness=int((box_thickness-0.5)/fdtd_grid))
clad_lower_layer = hwc.Layer(jnp.zeros((nx, ny)), permittivity_values=n_sio2**2, layer_thickness=int(0.5/fdtd_grid))
clad_upper_layer = hwc.Layer(jnp.zeros((nx, ny)), permittivity_values=n_sio2**2, layer_thickness=int((clad_thickness-0.5)/fdtd_grid))
air_gap_layer = hwc.Layer(jnp.zeros((nx, ny)), permittivity_values=n_air**2, layer_thickness=int(0.3/fdtd_grid))
air_layer = hwc.Layer(jnp.zeros((nx, ny)), permittivity_values=n_air**2, layer_thickness=int(0.2/fdtd_grid))

print(f"Design layers: slab (h={h_slab}px, {slab_height*1000:.0f}nm), etch (h={h_etch}px, {etch_depth*1000:.0f}nm)")


## Step 3e: Assemble Structure

Stack all layers bottom-to-top into a 3D permittivity volume.


In [ ]:
structure = hwc.create_structure(
    layers=[substrate_layer, box_layer, slab_layer, etch_layer,
            clad_lower_layer, clad_upper_layer, air_gap_layer, air_layer],
)

Lx, Ly, Lz = structure.permittivity.shape[1], structure.permittivity.shape[2], structure.permittivity.shape[3]
print(f"Structure shape: ({Lx}, {Ly}, {Lz})")


## Step 3f: Visualize Layer Stack


In [ ]:
hwc.plot_structure(structure)


## Step 4: Cost Estimate


In [ ]:
# --- Cost estimate ---
# FDTD volume (from structure dimensions)
N_cells = Lx * Ly * Lz
timesteps = 20000         # FDTD timesteps per simulation (manual, for stability)
gcups = 30                # Hyperwave throughput: 30 billion cell-updates/sec

sim_time = N_cells * timesteps / (gcups * 1e9)  # seconds per simulation
step_time = sim_time * 2  # forward + adjoint per optimization step

print(f"Cost estimate:")
print(f"  FDTD volume:  {Lx} x {Ly} x {Lz} = {N_cells:,.0f} cells")
print(f"  Timesteps:    {timesteps:,} per simulation")
print(f"  Throughput:   {gcups} GCUPs")
print(f"  Per sim:      {N_cells:,.0f} * {timesteps:,} / {gcups}e9 = {sim_time:.0f}s")
print(f"  Per step:     {sim_time:.0f}s * 2 (fwd + adj) = {step_time:.0f}s (~{step_time/60:.0f} min)")
print(f"  Full pipeline (~400 steps): ~{step_time * 400 / 3600:.0f} GPU-hours")

## Step 5: Design Region


In [ ]:
# --- Initialize design region ---
# theta shape: (num_design_layers, nx, ny)
# Layer 0 = slab (fixed at 1.0), Layer 1 = etch (optimizable)
theta = np.zeros((2, nx, ny))
theta[0, :, :] = 1.0   # Slab layer: solid silicon everywhere
theta[1, :, :] = 0.5   # Etch layer: start at midpoint (grayscale)

# Add waveguide taper at output edge
taper_length = int(3.0 / theta_grid)  # 3um taper
taper_width_start = int(wg_width / theta_grid)
taper_width_end = int(12.0 / theta_grid)  # 12um grating width

for i in range(taper_length):
    frac = i / taper_length
    w = int(taper_width_start + frac * (taper_width_end - taper_width_start))
    y_start = ny // 2 - w // 2
    y_end = ny // 2 + w // 2
    theta[1, i, y_start:y_end] = 1.0  # Solid in taper region

print(f"Design region: {design_length} x {design_width} um")
print(f"Design pixels: {nx * ny:,}")
print(f"Etch layer initialized to 0.5 (grayscale)")

# Visualize the initial theta (etch layer)
hwc.plot_theta(theta[1])

## Step 6: Gaussian Source


In [ ]:
# Generate tilted Gaussian beam (SMF-28 fiber, 14.5 degree angle)
Lx, Ly, Lz = device.shape
freq = device.freq_band[0]
source_x = Lx // 2               # center of domain
source_y = Ly // 2
source_z = Lz - 30               # near top surface

source_field, input_power = hwc.generate_gaussian_source(
    sim_shape=device.shape,
    frequencies=np.array([freq]),
    source_pos=(source_x, source_y, source_z),
    waist_radius=int(beam_waist / fdtd_grid),   # 5.2um / 35nm = 149 pixels
    theta=fiber_angle * np.pi / 180,             # 14.5 deg in radians
    polarization='y',                             # TE polarization
    wavelength_um=wavelength,
    dx_um=fdtd_grid,
    gpu_type="B200",
)
print(f"Source generated. Input power: {float(input_power):.2f}")
print(f"Source field shape: {source_field.shape}")

# Visualize source field
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
mid_z = source_field.shape[4] // 2
ax1.imshow(np.abs(source_field[0, 1, :, :, mid_z])**2)
ax1.set_title("|Ey|^2 (XY plane)")
mid_y = source_field.shape[3] // 2
ax2.imshow(np.real(source_field[0, 1, :, mid_y, :]).T, aspect="auto")
ax2.set_title("Re(Ey) (XZ plane)")
plt.tight_layout()
plt.show()

## Step 7: Waveguide Mode


In [ ]:
# Solve TE0 fundamental mode of 500nm x 220nm Si waveguide
mode_field, n_eff = hwc.solve_waveguide_mode(
    grid=fdtd_grid,
    waveguide_width=wg_width,     # 0.5 um
    waveguide_height=wg_height,   # 0.220 um
    n_core=n_si,
    n_clad=n_sio2,
    wavelength=wavelength,
    mode_number=0,                # fundamental TE
)
print(f"TE0 effective index: {n_eff:.4f}")
print(f"Mode confinement: n_eff > n_clad ({n_eff:.4f} > {n_sio2})")

# Visualize the waveguide mode
hwc.plot_mode(mode_field, beta=n_eff, mode_num=0)

## Step 8: Objective Function


In [ ]:
# Define the optimization objective using expression trees
# Mode coupling: maximize overlap with the target waveguide mode
objective = hwc.objectives.mode_coupling(
    mode_field=mode_field,
    input_power=float(input_power),
    mode_cross_power=1.0,            # mode self-overlap power
    monitor="waveguide_output",
)

# Expression trees compose with standard math operators:
# Broadband: maximize worst-case across wavelengths
# e1 = hwc.objectives.mode_coupling(mode_field, P_in, P_m, "wg", freq_idx=0)
# e2 = hwc.objectives.mode_coupling(mode_field, P_in, P_m, "wg", freq_idx=1)
# e3 = hwc.objectives.mode_coupling(mode_field, P_in, P_m, "wg", freq_idx=2)
# objective = hwc.objectives.min_of(e1, e2, e3)

# Field-level objectives:
# ey = hwc.objectives.field("Ey", "focus_monitor")
# objective = hwc.objectives.sum_spatial(hwc.objectives.abs_val(ey))

print(f"Objective: mode coupling efficiency")
print(f"Type: {type(objective).__name__}")

## Step 9: Phase 1: Freeform Optimization


In [ ]:
# Phase 1: Freeform topology optimization
# Request up to 100 steps. We can stop the run early if efficiency plateaus.
results_freeform = hwc.optimize(
    layers=layers,
    theta={"slab": theta_slab, "etch": theta_etch},
    grid=fdtd_grid,
    wavelength=wavelength,
    source=source_field,
    mode=mode_field,
    phase="freeform",
    n_steps=100,
)

# Plot results
hwc.plot_phase_summary(results_freeform, title="Freeform", show_fields=True)

## Step 10: Phase 2: Binarization


In [ ]:
# Strategy A: Continuous ramp (used here for the GC)
results_binarize = hwc.optimize(
    layers=layers,
    grid=fdtd_grid,
    wavelength=wavelength,
    initial_design=results_freeform.design,
    source=source_field,
    mode=mode_field,
    phase="binarize",
    n_steps=200,
    beta_init=4.0,        # start soft
    beta_max=64.0,        # end sharp
    learning_rate=0.01,
)

# Strategy B: Stepped periods (more conservative)
# result = results_freeform
# for beta in [8, 16, 32, 64]:
#     result = hwc.optimize(
#         device, source, mode,
#         phase="binarize",
#         initial_design=result.design,
#         n_steps=25,
#         beta_init=beta,
#         beta_max=beta,    # flat -- no ramp within each period
#         learning_rate=0.01,
#     )

hwc.plot_phase_summary(results_binarize, title="Binarization")

## Step 11: Phase 3: Fabrication Constraints


In [ ]:
# Phase 3: Apply fabrication constraints
results_dfm = hwc.optimize(
    layers=layers,
    grid=fdtd_grid,
    wavelength=wavelength,
    initial_design=results_binarize.design,
    source=source_field,
    mode=mode_field,
    phase="dfm",
    n_steps=100,
    disk_radius=3,             # min feature + min gap = (2*3+1)*17.5nm = 122nm
)

# Plot results
hwc.plot_phase_summary(results_dfm, title="DFM Recovery", show_fields=True)

## Step 12: Phase 4: Surgery and Recovery


In [ ]:
# Phase 4a: Geometric surgery (remove small features, fill small holes)
design_clean = hwc.surgery(
    design=results_dfm.design,
    min_feature_size=0.105,
)
print(f"Surgery: removed {design_clean.removed_islands} islands, "
      f"filled {design_clean.filled_holes} holes")
print(f"Post-surgery efficiency: {design_clean.efficiency*100:.1f}%")

# Phase 4b: Recovery optimization - first 30 steps
results_recovery_1 = hwc.optimize(
    layers=layers,
    grid=fdtd_grid,
    wavelength=wavelength,
    initial_design=design_clean,
    source=source_field,
    mode=mode_field,
    phase="recovery",
    n_steps=30,
    disk_radius=3,
)

## Step 13: Phase 4 (continued): Recovery


In [ ]:
# Phase 4c: Continue recovery from checkpoint - another 30 steps
results_final = hwc.optimize(
    layers=layers,
    grid=fdtd_grid,
    wavelength=wavelength,
    initial_design=results_recovery_1.design,
    source=source_field,
    mode=mode_field,
    phase="recovery",
    n_steps=30,
    disk_radius=3,
)

# Plot results
hwc.plot_phase_summary(results_final, title="Recovery", show_fields=True)

## Step 14: DRC Verification


In [ ]:
# Run DRC check on final design
# Uses morphological opening with disk(r=3)
# At 17.5nm/px: disk diameter = (2*3+1) * 17.5nm = 122.5nm
drc_report = hwc.check_drc(
    design=results_final.design,
    disk_radius=3,            # disk(r=3), ~122nm at 17.5nm/px
    pixel_size=0.0175,        # um/px
)

print(f"CD violations:  {drc_report.cd_pct:.2f}%")
print(f"Gap violations: {drc_report.gap_pct:.2f}%")
print(f"Disk:           disk(r={drc_report.disk_radius}) = {drc_report.min_feature_nm:.0f}nm")
print(f"Binarization:   {drc_report.binarization_score*100:.1f}%")
print(f"Status:         {drc_report.status}")

# Visualize the final design with DRC result
hwc.plot_theta(results_final.design.theta)

## Step 15: Pipeline Summary


In [ ]:
# Generate full pipeline summary
hwc.plot_pipeline_summary(
    phases=[results_freeform, results_binarize, results_dfm, results_final],
    labels=["Freeform", "Binarization", "DFM", "Recovery"],
)

## Step 16: GDS Export


In [ ]:
# Export optimized design to GDSII
gds_path = hwc.export_gds(
    design=results_final.design,
    filename="gc_inverse_designed.gds",
    layer=(1, 0),             # GDS layer for etch
    pixel_size=0.0175,        # um per pixel
)

print(f"GDS exported: {gds_path}")
print(f"Design size: {results_final.design.shape[0] * 0.0175:.1f} x "
      f"{results_final.design.shape[1] * 0.0175:.1f} um")

# Visualize the exported GDS layout
hwc.plot_gds(gds_path)